<a href="https://colab.research.google.com/github/juanarrow/BigData-notebooks/blob/main/sesion-12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sesión 12 - Metodologías de Minería de Datos: CRISP-DM y SEMMA

**Módulo:** Sistemas de Big Data (SBD - 5074)

**Contenido:** 2.4 Métodos y algoritmos de minería de datos (SEMMA, CRISP-DM) — Proyecto práctico: Predicción de Churn

---

## Objetivos
- Comprender las fases de la metodología CRISP-DM
- Aplicar el modelo SEMMA en proyectos de datos
- Comparar ambas metodologías y seleccionar la más adecuada
- Aplicar CRISP-DM de forma completa en un proyecto real de Machine Learning

## Metodologías de Minería de Datos

### CRISP-DM vs SEMMA

Las dos metodologías más utilizadas en proyectos de ciencia de datos son **CRISP-DM** y **SEMMA**.

#### CRISP-DM (Cross-Industry Standard Process for Data Mining)

Proceso estándar y cíclico que cubre desde el entendimiento del negocio hasta el despliegue:

```
Business Understanding → Data Understanding → Data Preparation
        ↑                                              ↓
   Deployment      ←      Evaluation      ←      Modeling
```

| Fase | Objetivo | % Tiempo |
|------|----------|----------|
| **1. Business Understanding** | Entender el problema de negocio | 10% |
| **2. Data Understanding** | Conocer y explorar los datos | 20% |
| **3. Data Preparation** | Limpiar y transformar los datos | 40% |
| **4. Modeling** | Crear y entrenar modelos | 15% |
| **5. Evaluation** | Validar resultados con el negocio | 10% |
| **6. Deployment** | Poner el modelo en producción | 5% |

---

#### SEMMA (Sample, Explore, Modify, Model, Assess) — SAS Institute

Metodología más técnica y centrada en el análisis, sin incluir explícitamente el entendimiento del negocio ni el despliegue:

```
SAMPLE → EXPLORE → MODIFY → MODEL → ASSESS
```

| Fase | Descripción |
|------|-------------|
| **Sample** | Extraer una muestra representativa del dataset |
| **Explore** | Análisis exploratorio y visualización |
| **Modify** | Transformación, creación de variables y limpieza |
| **Model** | Construcción y entrenamiento de modelos |
| **Assess** | Evaluación de la calidad y rendimiento del modelo |

---

#### Comparativa CRISP-DM vs SEMMA

| Aspecto | CRISP-DM | SEMMA |
|---------|----------|-------|
| **Origen** | Consorcio industria (IBM, NCR, SPSS...) | SAS Institute |
| **Enfoque** | Proceso completo del proyecto | Técnico/Analítico |
| **Entendimiento negocio** | ✅ Fase 1 explícita | ❌ Se asume como previo |
| **Despliegue** | ✅ Fase 6 explícita | ❌ No incluido |
| **Iterativo** | ✅ Muy iterativo y cíclico | ⚠️ Más lineal |
| **Documentación** | ✅ Énfasis en entregables | ⚠️ Menos estructurada |
| **Uso actual** | Más extendido en industria | Entorno SAS/académico |

> **¿Cuándo usar cada una?**
> - **CRISP-DM**: Proyectos empresariales completos donde el contexto de negocio es clave.
> - **SEMMA**: Análisis técnicos donde el problema ya está bien definido por otros equipos.

---

### Caso práctico: CRISP-DM aplicado — Predicción de Churn

A continuación aplicamos las 6 fases de CRISP-DM a un proyecto real de predicción de abandono de clientes.

## 1. Generación de datos simulados

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import plotly.express as px
import plotly.graph_objects as go

# ========================================
# GENERAR DATOS SIMULADOS
# ========================================
np.random.seed(42)
n_clientes = 1000

df = pd.DataFrame({
    'cliente_id': range(1, n_clientes + 1),
    'antiguedad_meses': np.random.randint(1, 72, n_clientes),
    'gasto_mensual': np.random.normal(100, 30, n_clientes).clip(20, 200),
    'num_productos': np.random.randint(1, 5, n_clientes),
    'llamadas_soporte': np.random.poisson(2, n_clientes),
    'satisfaccion': np.random.randint(1, 6, n_clientes),
    'tiene_contrato': np.random.choice([0, 1], n_clientes, p=[0.3, 0.7]),
})

# Crear variable objetivo (churn) basada en lógica de negocio
prob_churn = (
    (df['satisfaccion'] < 3) * 0.3 +
    (df['llamadas_soporte'] > 3) * 0.2 +
    (df['antiguedad_meses'] < 12) * 0.15 +
    (df['tiene_contrato'] == 0) * 0.2 +
    np.random.random(n_clientes) * 0.15
)
df['churn'] = (prob_churn > 0.5).astype(int)

print(f"Dataset: {len(df)} clientes")
print(f"Tasa de churn: {df['churn'].mean():.1%}")
print("\nPrimeras filas:")
print(df.head())

## 2. Análisis exploratorio

In [ ]:
# Análisis por variable
fig = px.histogram(df, x='satisfaccion', color='churn', barmode='group',
                   title='Distribución de Satisfacción por Churn',
                   labels={'churn': 'Abandono'})
fig.show()

In [ ]:
# Box plot de gasto por churn
fig = px.box(df, x='churn', y='gasto_mensual', 
             title='Gasto Mensual por Estado de Churn',
             labels={'churn': 'Abandono', 'gasto_mensual': 'Gasto Mensual (€)'})
fig.show()

## 3. Preparación de datos y entrenamiento

In [ ]:
# ========================================
# PREPARACIÓN DE DATOS
# ========================================

# Features y target
X = df.drop(['cliente_id', 'churn'], axis=1)
y = df['churn']

# Split train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Escalar datos
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Train: {len(X_train)} | Test: {len(X_test)}")
print(f"Features: {X.columns.tolist()}")

In [ ]:
# ========================================
# ENTRENAR MODELO
# ========================================

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_scaled, y_train)

# Predicciones
y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:, 1]

print("=== RESULTADOS ===")
print(classification_report(y_test, y_pred, target_names=['No Churn', 'Churn']))

In [ ]:
# AUC-ROC
auc = roc_auc_score(y_test, y_prob)
print(f"\nAUC-ROC: {auc:.3f}")

# Matriz de confusión
cm = confusion_matrix(y_test, y_pred)
fig = px.imshow(cm, text_auto=True, 
                labels=dict(x="Predicción", y="Real"),
                x=['No Churn', 'Churn'],
                y=['No Churn', 'Churn'],
                title='Matriz de Confusión')
fig.show()

## 4. Importancia de variables

In [ ]:
# Importancia de features
importances = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=True)

fig = px.bar(importances, x='importance', y='feature', orientation='h',
             title='Importancia de Variables para Predicción de Churn')
fig.show()

print("\n=== HALLAZGOS CLAVE ===")
print("1. SATISFACCIÓN es el factor más importante")
print("2. LLAMADAS A SOPORTE indica frustración")
print("3. CONTRATO reduce significativamente el churn")

---

## Recursos adicionales

- [Scikit-learn Documentation](https://scikit-learn.org/)
- [CRISP-DM Methodology](https://www.datascience-pm.com/crisp-dm-2/)